# Lab 1 — Intro to Behavioral TV Data
Name: Anrew Bohdan  | Group: КМ-43

**Rules for this lab (so everyone gets the same numbers):**
- a household or person has watched a telecast if they watched at least 1 minute of it;
- persons are R and L only; S guests are left out of every person number;
- a telecast's rating = sum of the weights of those who watched it ÷ sum of the weights of everyone in-tab that broadcast day (in the same group) × 100;
- for reach over several days, each person gets one weight for the month: their average weight over the broadcast days they were in-tab. Reach = sum of these weights over people who watched at least once ÷ sum over everyone in the group × 100.

**Output format:** plain values (no units); percentages 0–100 with one decimal; shares 0–1 with three decimals; audiences in millions with two decimals.

## Setup

In [46]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab1-tv-panel/versions/2") # Colab and Kaggle
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")

Using Colab cache for faster access to the 'adtech-lab1-tv-panel' dataset.


## Task 1.1 — Panel and universe
How many telecasts, distinct programs and networks are in `programs`? How many households and persons were in-tab at least once in October (use `daily_weights`)? Why does `viewer_attributes` list a few more? On an average day, how many are in-tab, and how many US households and persons do their weights add up to?

In [67]:
# Your code here
count_telecast  = int(programs["telecast_id"].count())
distinct_program = programs["program_name"].nunique()
distinct_network = programs["network_name"].nunique()
hh = daily_weights[daily_weights["resp_type"]=="H"]
pp = daily_weights[daily_weights["resp_type"].isin(["R","L"])]
pp_distinct = pp["respondent_id"].nunique()
hh_distinct = hh["respondent_id"].nunique()
per_day_hh = hh.groupby("broadcast_date")["weight"].agg(["size", "sum"])
hh_in_tab_avg=per_day_hh["size"].mean().round(2)
per_day_pp = pp.groupby("broadcast_date")["weight"].agg(["size", "sum"])
pp_in_tab_avg  =per_day_pp["size"].mean().round(2)
us_tv_households  = (per_day_hh["sum"].mean() / 1e6).round(2)
us_persons = (per_day_pp["sum"].mean() / 1e6).round(2)
print("Telecasts:", count_telecast)
print("Programs:", distinct_program)
print("Networks:", distinct_network)
print("Households in panel:", hh_distinct)
print("Persons in panel:", pp_distinct)
print("Households in-tab, average day:", hh_in_tab_avg)
print("Persons in-tab, average day:", pp_in_tab_avg )
print("US TV households represented, average day (M):", us_tv_households)
print("US persons represented, average day (M):", us_persons)

Telecasts: 5841
Programs: 246
Networks: 10
Households in panel: 4254
Persons in panel: 10210
Households in-tab, average day: 3878.45
Persons in-tab, average day: 9268.68
US TV households represented, average day (M): 125.8
US persons represented, average day (M): 317.28


In [69]:
va_hh = viewer_attributes[viewer_attributes["attr_id"] == 104]["respondent_id"].nunique()
va_pp = viewer_attributes[viewer_attributes["attr_id"].isin([101, 102])]["respondent_id"].nunique()
print("Extra households in viewer_attributes:", va_hh - hh_distinct)
print("Extra persons in viewer_attributes:", va_pp - pp_distinct)

Extra households in viewer_attributes: 2
Extra persons in viewer_attributes: 6


In [ ]:
va = viewer_attributes.merge(attributes, on="attr_id", how="left")
display(va.head(10))
display(attributes.groupby("attr_group")["attr_name"].apply(lambda s: ", ".join(s.astype(str))).to_frame())

**Answer:** `viewer_attributes` lists 2 more households and 6 more persons than `daily_weights`. These respondents have a profile but were never in-tab in October. They may have just joined or already left the panel, or their meter failed, so they were never measured during the month.

## Task 1.2 — Average audience by hour of day
Plot the projected average audience (millions of persons watching these ten networks per minute) by hour of day, separately for weekdays and weekends. Which hour is the peak? How does the weekend differ?

*Hint:* a session from 20:50 to 21:20 adds 10 minutes to hour 20 and 20 minutes to hour 21. Split sessions at hour boundaries (or expand them into minutes) and multiply by the weight before summing. Average audience in an hour = weighted minutes in that hour ÷ 60 ÷ number of such days. Count weekdays and weekends by broadcast day: Friday 01:00 still belongs to Thursday's broadcast day.

In [ ]:
def to_naive(s):
    s = pd.to_datetime(s)
    if getattr(s.dt, "tz", None) is not None:
        s = s.dt.tz_localize(None)
    return s


for col in ["start_ts", "end_ts", "broadcast_date"]:
    programs[col] = to_naive(programs[col])
for col in ["start_ts", "end_ts"]:
    viewings[col] = to_naive(viewings[col])
daily_weights["broadcast_date"] = to_naive(daily_weights["broadcast_date"]).dt.normalize()
programs["broadcast_date"] = programs["broadcast_date"].dt.normalize()
viewings = viewings.reset_index(drop=True)

# person sessions only (R and L; guests are left out)
pv = viewings[viewings["resp_type"].isin(["R", "L"])].copy()
pv = pv[pv["end_ts"] > pv["start_ts"]]

# split every session at hour boundaries
pv["h0"] = pv["start_ts"].dt.floor("h")
pv["n"] = np.ceil((pv["end_ts"] - pv["h0"]) / pd.Timedelta("1h")).astype(int)
seg = pv.loc[pv.index.repeat(pv["n"]), ["start_ts", "end_ts", "weight", "h0"]]
seg["k"] = seg.groupby(level=0).cumcount()
seg["hour_start"] = seg["h0"] + pd.to_timedelta(seg["k"], unit="h")
seg_start = seg["start_ts"].where(seg["start_ts"] > seg["hour_start"], seg["hour_start"])
hour_end = seg["hour_start"] + pd.Timedelta("1h")
seg_end = seg["end_ts"].where(seg["end_ts"] < hour_end, hour_end)
seg["minutes"] = (seg_end - seg_start).dt.total_seconds() / 60
seg["wmin"] = seg["minutes"] * seg["weight"]

# broadcast day (06:00-05:59) decides weekday vs weekend
seg["bdate"] = (seg["hour_start"] - pd.Timedelta("6h")).dt.normalize()
october_days = pd.Series(sorted(programs["broadcast_date"].unique()))
seg = seg[seg["bdate"].isin(october_days)]
seg["weekend"] = seg["bdate"].dt.dayofweek >= 5
seg["hour"] = seg["hour_start"].dt.hour

n_days = (october_days.dt.dayofweek >= 5).value_counts()   # False = weekdays, True = weekends
aud = seg.groupby(["hour", "weekend"])["wmin"].sum().unstack("weekend")
for is_we in [False, True]:
    aud[is_we] = aud[is_we] / 60 / n_days[is_we] / 1e6
aud.columns = ["Weekdays" if not c else "Weekends" for c in aud.columns]

order = list(range(6, 24)) + list(range(0, 6))   # broadcast-day order
aud_plot = aud.reindex(order)
fig, ax = plt.subplots(figsize=(11, 4.5))
for col, color in [("Weekdays", "#1f5aa6"), ("Weekends", "#e07b00")]:
    ax.plot(range(24), aud_plot[col].values, marker="o", ms=4, color=color, label=col)
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}" for h in order])
ax.set_xlabel("Clock hour (US Eastern), broadcast day 06:00-05:59")
ax.set_ylabel("Average audience, M persons")
ax.set_title("Projected average audience of the ten networks by hour")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

print("Weekdays:", n_days[False], "| Weekends:", n_days[True])
print("Peak hour, weekdays:", int(aud["Weekdays"].idxmax()))
print("Peak hour, weekends:", int(aud["Weekends"].idxmax()))

**Answer:** On both weekdays and weekends viewing peaks in prime time (the evening hours, see the peak hours printed above), when most people are at home. On weekends the curve is flatter: people are free during the day, so daytime and afternoon viewing is higher than on weekdays (on weekend afternoons live sports also add viewers), while the weekday curve rises more sharply only in the evening. Overnight hours are the lowest on both curves.

## Task 1.3 — Session length
Plot the distribution of person session length, weighting each session by its weight. Report the weighted mean and median. Why are they different, and which one would you put in a client deck?

In [ ]:
def wquantile(x, w, q):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    order = np.argsort(x)
    x, w = x[order], w[order]
    cum = np.cumsum(w) / w.sum()
    return x[np.searchsorted(cum, q - 1e-12)]


pv["length"] = (pv["end_ts"] - pv["start_ts"]).dt.total_seconds() / 60
w_mean = np.average(pv["length"], weights=pv["weight"])
w_median = wquantile(pv["length"], pv["weight"], 0.5)

fig, ax = plt.subplots(figsize=(10, 4.5))
clip = 240
ax.hist(pv["length"].clip(upper=clip), bins=np.arange(0, clip + 5, 5),
        weights=pv["weight"] / 1e6, color="#1f5aa6", alpha=0.85)
ax.axvline(w_median, color="#e07b00", lw=2, label=f"Median {w_median:.1f} min")
ax.axvline(w_mean, color="#c0392b", lw=2, ls="--", label=f"Mean {w_mean:.1f} min")
ax.set_xlabel(f"Session length, minutes (sessions over {clip} min shown in the last bin)")
ax.set_ylabel("Projected sessions, M")
ax.set_title("Weighted distribution of person session length")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

print("Session length, weighted mean (min):", f"{w_mean:.1f}")
print("Session length, weighted median (min):", f"{w_median:.1f}")

**Answer:** The distribution is strongly right-skewed: most sessions are short (people zap between channels or watch a few minutes), but a small number of very long sessions (a whole evening, a game, or a TV left on) pull the mean up. That is why the mean is clearly higher than the median. For a client deck I would show the median, because it describes a typical session; the mean is driven by a few extreme sessions and would overstate how long a usual viewer stays.

## Task 2 — Program reach curve
Take all October telecasts of *CBS Evening News* (weekdays on CBS).
1. For each telecast compute the household rating, the P2+ rating and the projected audience in millions of persons.
2. In air order, compute the cumulative reach after each telecast for P2+, A18–34 and A18–49, and plot the three curves.
3. Why does cumulative reach keep growing while each telecast's rating stays roughly flat? Which audience does the program reach best, and does that match what you know about who watches network evening news?

In [ ]:
code_name = attributes.set_index("attr_id")["attr_name"]
person_ids = va.loc[va["attr_id"].isin([101, 102]), "respondent_id"].unique()

age = (va[va["attr_id"] == 300][["respondent_id", "value"]]
       .rename(columns={"value": "age"}).drop_duplicates("respondent_id"))
sex = va[va["attr_id"].isin([201, 202])][["respondent_id", "attr_id"]].drop_duplicates("respondent_id")
sex["sex"] = sex["attr_id"].map(code_name).astype(str).str.strip().str[0].str.upper().replace({"F": "W"})

persons = (pd.DataFrame({"respondent_id": person_ids})
           .merge(age, on="respondent_id", how="left")
           .merge(sex[["respondent_id", "sex"]], on="respondent_id", how="left"))
print("Sex codes:", {c: code_name.get(c) for c in [201, 202]})
print("Persons with attributes:", len(persons), "| missing age:", persons["age"].isna().sum(),
      "| missing sex:", persons["sex"].isna().sum())
display(persons.head())

# monthly weight of every person = average daily weight over the in-tab days
monthly_w = (daily_weights[daily_weights["resp_type"].isin(["R", "L"])]
             .groupby("respondent_id")["weight"].mean())

# audiences (demographic groups)
AUDIENCES = {
    "P2+": (2, 200, None), "A18-34": (18, 34, None), "A18-49": (18, 49, None),
    "M18-34": (18, 34, "M"), "W18-34": (18, 34, "W"), "M35-54": (35, 54, "M"),
    "W35-54": (35, 54, "W"), "M55+": (55, 200, "M"), "W55+": (55, 200, "W"),
}


def members(aud):
    lo, hi, sx = AUDIENCES[aud]
    p = persons[persons["age"].between(lo, hi)]
    if sx is not None:
        p = p[p["sex"] == sx]
    return set(p["respondent_id"])


def telecast_viewers(tel, rtypes):
    """Who watched each telecast for at least 1 minute (sum over sessions)."""
    v = viewings[viewings["resp_type"].isin(rtypes)
                 & viewings["network_code"].isin(tel["network_code"].unique())]
    v = v[(v["end_ts"] > tel["start_ts"].min()) & (v["start_ts"] < tel["end_ts"].max())]
    m = v[["respondent_id", "network_code", "start_ts", "end_ts"]].merge(
        tel[["telecast_id", "network_code", "start_ts", "end_ts", "broadcast_date"]],
        on="network_code", suffixes=("", "_t"))
    m = m[(m["end_ts"] > m["start_ts_t"]) & (m["start_ts"] < m["end_ts_t"])]
    s = m["start_ts"].where(m["start_ts"] > m["start_ts_t"], m["start_ts_t"])
    e = m["end_ts"].where(m["end_ts"] < m["end_ts_t"], m["end_ts_t"])
    m["overlap"] = (e - s).dt.total_seconds() / 60
    w = m.groupby(["telecast_id", "broadcast_date", "respondent_id"], as_index=False)["overlap"].sum()
    return w[w["overlap"] >= 1]


def telecast_ratings(tel, rtypes, ids=None):
    """Rating (%) and projected audience per telecast for one group, with daily weights."""
    dw = daily_weights[daily_weights["resp_type"].isin(rtypes)]
    if ids is not None:
        dw = dw[dw["respondent_id"].isin(ids)]
    universe = dw.groupby("broadcast_date")["weight"].sum()
    vw = telecast_viewers(tel, rtypes)
    if ids is not None:
        vw = vw[vw["respondent_id"].isin(ids)]
    vw = vw.merge(dw[["respondent_id", "broadcast_date", "weight"]],
                  on=["respondent_id", "broadcast_date"], how="inner")
    audience = vw.groupby("telecast_id")["weight"].sum().reindex(tel["telecast_id"]).fillna(0)
    rating = audience.values / universe.reindex(tel["broadcast_date"]).values * 100
    return pd.Series(rating, index=tel["telecast_id"]), audience


def cumulative_reach(tel, aud):
    """Cumulative reach (%) after each telecast in air order, with monthly weights."""
    ids = members(aud)
    w = monthly_w[monthly_w.index.isin(ids)]
    vw = telecast_viewers(tel, ["R", "L"])
    vw = vw[vw["respondent_id"].isin(w.index)]
    pos = pd.Series(range(len(tel)), index=tel["telecast_id"].values)
    first = vw.assign(pos=vw["telecast_id"].map(pos)).groupby("respondent_id")["pos"].min()
    gained = w.reindex(first.index).groupby(first.values).sum().reindex(range(len(tel)), fill_value=0)
    return gained.cumsum().values / w.sum() * 100

In [ ]:
cbs_news = programs[(programs["network_code"] == "CBS")
                    & programs["program_name"].str.contains("CBS Evening News", case=False, na=False)]
cbs_news = cbs_news.sort_values("start_ts").reset_index(drop=True)
print("Program names found:", list(cbs_news["program_name"].unique()))

hh_rating, _ = telecast_ratings(cbs_news, ["H"])
p2_rating, p2_aud = telecast_ratings(cbs_news, ["R", "L"], members("P2+"))

table2 = pd.DataFrame({
    "date": cbs_news["start_ts"].dt.strftime("%a %Y-%m-%d %H:%M").values,
    "HH rating (%)": hh_rating.values.round(1),
    "P2+ rating (%)": p2_rating.values.round(1),
    "Audience (M)": (p2_aud.values / 1e6).round(2),
}, index=cbs_news["telecast_id"].values)
display(table2)

In [ ]:
reach = {aud: cumulative_reach(cbs_news, aud) for aud in ["P2+", "A18-34", "A18-49"]}

fig, ax = plt.subplots(figsize=(10, 4.5))
x = range(1, len(cbs_news) + 1)
for (aud, r), color in zip(reach.items(), ["#1f5aa6", "#e07b00", "#2e8b57"]):
    ax.plot(x, r, marker="o", ms=4, color=color, label=aud)
ax.set_xlabel("Telecasts aired (October, in air order)")
ax.set_ylabel("Cumulative reach, %")
ax.set_title("CBS Evening News: cumulative reach")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

print("Telecasts of the program:", len(cbs_news))
print("Average household rating (%):", f"{hh_rating.mean():.1f}")
print("Average P2+ rating (%):", f"{p2_rating.mean():.1f}")
print("Average projected audience (M):", f"{p2_aud.mean() / 1e6:.2f}")
print("Final reach P2+ (%):", f"{reach['P2+'][-1]:.1f}")
print("Final reach A18-34 (%):", f"{reach['A18-34'][-1]:.1f}")
print("Final reach A18-49 (%):", f"{reach['A18-49'][-1]:.1f}")

**Answer:** A rating measures one telecast: what share of the audience watched that single episode, and since the program has a steady audience every evening, it stays roughly flat. Reach is cumulative: it counts everyone who watched at least once, so every new telecast adds people who had not watched before (occasional viewers), while loyal viewers are counted only once. That is why the curve keeps growing but more and more slowly. The program reaches P2+ best and young adults (A18-34) worst, with A18-49 in between. This matches what is known about network evening news: its audience is mostly older people, who watch TV news at fixed times, while young adults get their news online.

## Task 3 — Program frequency
For *The Young and the Restless* (a daily soap on CBS), count for every person who watched how many of its October telecasts they saw: that is the program's frequency. Use each person's monthly weight from the rules.
1. Report the projected number of viewers (millions), and the weighted average and median frequency.
2. Plot the frequency distribution in projected viewers: how many millions watched 1, 2, 3, … telecasts.
3. What share of all projected views comes from the most loyal viewers? Define them as everyone whose frequency is at or above the weighted 80th percentile of frequency (whole frequency levels, no splitting). Is the average frequency a fair summary ("the average can lie")?

In [ ]:
yr = programs[(programs["network_code"] == "CBS")
              & programs["program_name"].str.contains("Young and the Restless", case=False, na=False)]
yr = yr.sort_values("start_ts").reset_index(drop=True)
print("Program names found:", list(yr["program_name"].unique()), "| telecasts:", len(yr))

vw_yr = telecast_viewers(yr, ["R", "L"])
freq = vw_yr.groupby("respondent_id")["telecast_id"].nunique()
w_yr = monthly_w.reindex(freq.index)
freq, w_yr = freq[w_yr.notna()], w_yr.dropna()

viewers_m = w_yr.sum() / 1e6
avg_freq = np.average(freq, weights=w_yr)
med_freq = wquantile(freq, w_yr, 0.5)
p80 = wquantile(freq, w_yr, 0.8)
loyal = freq >= p80
loyal_share = (freq[loyal] * w_yr[loyal]).sum() / (freq * w_yr).sum()

dist = (w_yr.groupby(freq).sum() / 1e6).reindex(range(1, len(yr) + 1), fill_value=0)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(dist.index, dist.values, color=["#e07b00" if f >= p80 else "#1f5aa6" for f in dist.index])
ax.set_xticks(dist.index)
ax.set_xlabel("Number of October telecasts watched (frequency)")
ax.set_ylabel("Projected viewers, M")
ax.set_title(f"The Young and the Restless: frequency distribution (orange = most loyal, frequency >= {p80:.0f})")
ax.grid(alpha=0.3, axis="y")
plt.show()

print("Projected viewers (M):", f"{viewers_m:.2f}")
print("Average frequency (weighted):", f"{avg_freq:.1f}")
print("Median frequency (weighted):", f"{med_freq:.1f}")
print("Weighted 80th percentile of frequency:", int(p80))
print("Share of views from the most loyal viewers:", f"{loyal_share:.3f}")

**Answer:** No, the average frequency alone is not a fair summary. The distribution is not bell-shaped: many people watched only one or a few episodes, while a loyal core watched almost every day. The average falls between these two groups and describes almost nobody. The most loyal viewers (at or above the weighted 80th percentile) are a small part of the audience but produce a disproportionately large share of all views (see the share above). For a daily soap, the frequency distribution, or at least the median plus the share of heavy viewers, tells much more than one average ("the average can lie").

## Task 4 — Networks by age and gender
For each network compute projected viewing minutes (minutes × weight) for six audiences: M18–34, W18–34, M35–54, W35–54, M55+ and W55+. List the top 5 networks for each audience.
- Which network has the youngest audience and which the oldest? Measure it as the share of each network's projected minutes coming from 18–34 viewers.
- Compute the same share once more without weights (raw panel minutes). Why is it different? What does it tell you about who is over- or under-represented in the panel, and what would go wrong if you reported raw panel numbers to a client?
- **Bonus (+1 point):** if you sold ads on the oldest network, how would you pitch it to an advertiser who wants young adults?

In [ ]:
pv4 = pv.merge(persons, on="respondent_id", how="left")
pv4["wmin"] = pv4["length"] * pv4["weight"]
pv4["band"] = pd.cut(pv4["age"], bins=[17, 34, 54, 200], labels=["18-34", "35-54", "55+"])
pv4["audience"] = pv4["sex"] + pv4["band"].astype(str)

six = ["M18-34", "W18-34", "M35-54", "W35-54", "M55+", "W55+"]
minutes = (pv4[pv4["audience"].isin(six)]
           .pivot_table(index="network_code", columns="audience", values="wmin", aggfunc="sum")
           .reindex(columns=six).fillna(0) / 1e6)

for a in six:
    top5 = minutes[a].nlargest(5).round(1).rename("Projected minutes (M)").to_frame()
    top5.index.name = f"Top 5 — {a}"
    display(top5)

In [ ]:
total_w = pv4.groupby("network_code")["wmin"].sum()
young_w = pv4[pv4["band"] == "18-34"].groupby("network_code")["wmin"].sum()
total_raw = pv4.groupby("network_code")["length"].sum()
young_raw = pv4[pv4["band"] == "18-34"].groupby("network_code")["length"].sum()

share = pd.DataFrame({
    "18-34 share, weighted": (young_w / total_w).reindex(total_w.index).fillna(0),
    "18-34 share, raw": (young_raw / total_raw).reindex(total_w.index).fillna(0),
})
share["difference (raw - weighted)"] = share["18-34 share, raw"] - share["18-34 share, weighted"]
share = share.sort_values("18-34 share, weighted", ascending=False).round(3)
display(share)

print("Youngest network:", share["18-34 share, weighted"].idxmax())
print("Oldest network:", share["18-34 share, weighted"].idxmin())

**Answer:** The raw share counts every panel member equally, but the panel does not mirror the US population exactly: some groups are recruited more easily than others. The weights fix this by making each person stand for the right number of Americans. Where the raw 18-34 share is lower than the weighted one, young adults are under-represented in the panel and the weights scale them up (and older viewers, who are easier to recruit, are over-represented). If we reported raw panel numbers to a client, the networks' audiences would look older than they really are, young-skewing networks would be undervalued, and the client would plan and pay for the wrong audience.

**Bonus answer:** The oldest network has a big, loyal and attentive audience, so even its small 18-34 share is a real number of young adults, and they are reached in a premium, brand-safe environment. Adding it to a campaign built on young-skewing networks gives extra reach and frequency among young adults at a lower price per viewer, plus their parents, who often influence or pay for the purchase.